# 03 -- XAI instruments walkthrough

Narrative only (`project.md` SS9): reads a single committed run's `xai/*.npz` files
(Poisson, `q_serial`) and shows what each of the 6 core instruments actually recorded,
at the same final checkpoint, the same way `scripts/adjudicate_predictions.py` and
`scripts/make_figures.py` do. No live model, no GPU.

In [1]:
import json
from pathlib import Path
import numpy as np
from qapinn.runner import enumerate_runs

REPO_ROOT = Path.cwd() if (Path.cwd() / "configs").is_dir() else Path.cwd().parent
cfgs = enumerate_runs(REPO_ROOT / "configs" / "exp" / "core_matrix.yaml")
cfg = next(c for c in cfgs if c.pde.name == "poisson" and c.model.family == "q_serial")
run_dir = REPO_ROOT / "results" / "runs" / cfg.run_id
xai_dir = run_dir / "xai"
metrics = json.loads((run_dir / "metrics.json").read_text())
final_step = cfg.train.steps_adam + cfg.train.steps_lbfgs
print(f"run_dir: {run_dir.relative_to(REPO_ROOT).as_posix()}")
print(f"final rel_l2: {metrics['rel_l2']:.6f}, final step: {final_step}")

run_dir: results/runs/ce23b631c673
final rel_l2: 1.728254, final step: 22000


## NTK spectrum: eigenvalue decay at the final checkpoint

In [2]:
ntk = np.load(xai_dir / f"ntk_step{final_step}.npz")
print("ntk_step*.npz keys:", list(ntk.files))
eigs = np.sort(ntk["eigenvalues"])[::-1]
print(f"{len(eigs)} eigenvalues, top 5: {eigs[:5]}")
print(f"smallest 5 (should be >=0, PSD): {eigs[-5:]}")

ntk_step*.npz keys: ['eigenvalues', 'decay_exponent', 'condition_number', 'trace', 'effective_rank', 'drift']
128 eigenvalues, top 5: [4.97559221e+10 9.76887646e+09 1.20575505e+09 4.34546780e+07
 4.04705361e+06]
smallest 5 (should be >=0, PSD): [-7.16713679e-06 -7.63722905e-06 -8.57833001e-06 -9.04486412e-06
 -1.15049791e-05]


## Per-frequency error trajectory (spectral bias)

In [3]:
specerr = np.load(xai_dir / "specerr.npz")
print("specerr.npz keys:", list(specerr.files))
print(f"{specerr['errors'].shape[0]} checkpoints x {specerr['errors'].shape[1]} frequency bins")
print(f"checkpoints (training steps): {specerr['steps'].tolist()}")

specerr.npz keys: ['omega', 'errors', 'steps']
7 checkpoints x 1024 frequency bins
checkpoints (training steps): [0, 100, 500, 1000, 5000, 20000, 22000]


## Gradient variance (barren-plateau instrument)

In [4]:
gradvar = np.load(xai_dir / f"gradvar_step{final_step}.npz")
print("gradvar_step*.npz keys:", list(gradvar.files))
print(f"var_mean: {float(gradvar['var_mean']):.6g}")

gradvar_step*.npz keys: ['var_mean', 'var_per_param', 'n_qubits', 'n_layers']
var_mean: 2.03164e+14


## Fisher information / effective dimension

In [5]:
fisher = np.load(xai_dir / f"fisher_step{final_step}.npz")
print("fisher_step*.npz keys:", list(fisher.files))
print(f"effective_dimension: {float(fisher['effective_dimension']):.4f}")

fisher_step*.npz keys: ['eigenvalues', 'effective_dimension']
effective_dimension: 3.6159


## Attribution (integrated gradients) and layerwise probes

In [6]:
attribution = np.load(xai_dir / f"attribution_step{final_step}.npz")
probes = np.load(xai_dir / f"probes_step{final_step}.npz")
print("attribution_step*.npz keys:", list(attribution.files))
print(f"completeness_error: {float(attribution['completeness_error']):.6g}")
print("probes_step*.npz keys:", list(probes.files))

attribution_step*.npz keys: ['field', 'completeness_error', 'correlation']
completeness_error: 0.540584
probes_step*.npz keys: ['input', 'encoder', 'circuit', 'head']


## Summary

Every one of these instruments is run identically for every family/problem pair in the
core matrix (T1.12's own hook registry, wired into `train/loop.py`) -- this is what
makes the classical-vs-hybrid comparisons in `paper/sections/results.tex` and
`FINDINGS.md` an apples-to-apples measurement rather than two differently-instrumented
pipelines compared after the fact.